# Coverage and evidence audit · 1.4.1
Executed checks for source coverage, corrected likelihoods, copy removal and accepted numerical diagnostics. The formula and calibration panel are unchanged. Source-specific display evidence is not automatically admitted to the fit. AA private measurements are excluded from this notebook.

In [1]:
from pathlib import Path
import json, hashlib, math
root = Path.cwd()
while not (root / "data/index-config.yaml").exists():
    if root.parent == root: raise RuntimeError("Run inside the repository")
    root = root.parent
audit = root / "docs/audits/1.4.1-coverage"
summary = json.loads((audit / "audit-summary.json").read_text())
data = json.loads((audit / "accepted-input.json").read_text())
diagnostics = json.loads((audit / "accepted-diagnostics.json").read_text())
assert hashlib.sha256((audit / "accepted-input.json").read_bytes()).hexdigest() == summary["accepted_input_sha256"]
print(json.dumps({"before": summary["before"], "after": summary["after"], "newly_observed_models": summary["newly_observed_models"]}, indent=2))

{
  "before": {
    "site_evidence_rows": 1075,
    "models_with_evidence": 113,
    "distinct_model_benchmark_cells": 690,
    "catalog_models": 126,
    "benchmark_conditions": 27,
    "models_without_evidence": [
      "a-x-k2",
      "agnes-2-5-pro-alpha",
      "agnes-2-5-pro-beta",
      "apodex-1-1",
      "deepseek-v4-flash-vision-exp",
      "ling-3-0-flash",
      "mimo-v2-omni-0327",
      "motif-3",
      "muse-spark-1.3",
      "quasar-438b",
      "jt-4-1-flash-236b-a21b",
      "k2-horizon-375b-a23b",
      "qwen3.8-2.4t-a95b"
    ]
  },
  "after": {
    "site_evidence_rows": 1430,
    "models_with_evidence": 115,
    "distinct_model_benchmark_cells": 947,
    "catalog_models": 126,
    "benchmark_conditions": 94,
    "models_without_evidence": [
      "a-x-k2",
      "agnes-2-5-pro-alpha",
      "agnes-2-5-pro-beta",
      "apodex-1-1",
      "qwen3.8-2.4t-a95b",
      "jt-4-1-flash-236b-a21b",
      "k2-horizon-375b-a23b",
      "ling-3-0-flash",
      "mimo-v2-omni-03

In [2]:
assert diagnostics["accepted"] and diagnostics["divergences"] == 0
assert max(p["rhat"] for p in diagnostics["parameters"].values()) <= 1.01
assert min(min(p["ess_bulk"], p["ess_tail"]) for p in diagnostics["parameters"].values()) >= 400
assert min(diagnostics["ebfmi"]) >= .3
assert max(diagnostics["mcse_display_points"].values()) <= .3
for row in data["observations"]:
    if "variance" in row: assert math.isfinite(row["variance"]) and row["variance"] > 0
    if "x" in row: assert 0 <= row["x"] <= row["n_tasks"] * row["k_trials"]
assert all("simpleqa-verified" != b for b in data["benchmark_ids"])
assert all(b not in data["benchmark_ids"] for b in ["livebench-2026", "mrcr-v2-1m-8-needle", "osworld-2.0", "gpqa-diamond", "automationbench-public"])
math_index = data["benchmark_ids"].index("matharena-composite")
assert all(row["likelihood"] == "a_prime" for row in data["observations"] if row["benchmark_index"] == math_index)
print(json.dumps(summary["fit"], indent=2))

{
  "observations": 881,
  "models": 108,
  "systems": 127,
  "systems_with_direct_observations": 126,
  "benchmark_conditions": 18,
  "divergences": 0,
  "max_rhat": 1.0020242396862924,
  "min_ess": 1914.7655985135802,
  "min_ebfmi": 0.8578219675998788,
  "max_mcse": 0.2798268436131646,
  "retained_draws": 12000,
  "inference": {
    "chains": 4,
    "warmup": 3000,
    "samples": 5000,
    "target_accept": 0.995
  }
}


In [3]:
readiness = json.loads((audit / "readiness.json").read_text())
assert all(readiness["checks"].values())
corrections = json.loads((audit / "correction-inventory.json").read_text())
assert len({row["baseline_index"] for row in corrections}) == summary["integration"]["removed_unique_baseline_rows"]
print(f"{len(readiness['checks'])} preparation checks passed; {len({row['baseline_index'] for row in corrections})} baseline rows replaced/removed.")
print("Scope: source quality and numerical validity, not new out-of-sample predictive validation.")

18 preparation checks passed; 1003 baseline rows replaced/removed.
Scope: source quality and numerical validity, not new out-of-sample predictive validation.
